# 🚀 Notebook 04 - API com FastAPI

## Tech Challenge 4 - LSTM VALE3

---

### Objetivos deste Notebook:

1. **Criar a API** com FastAPI para servir o modelo
2. **Implementar endpoints** de previsão e saúde
3. **Testar localmente** antes do deploy
4. **Preparar estrutura** para AWS Lambda

---

### Arquitetura da API

```
Cliente (usuário/app)
        │
        ▼
┌─────────────────────────────────────┐
│           FastAPI                   │
│  ┌─────────────────────────────┐    │
│  │  GET /health                │    │  ← Verifica se API está ok
│  │  GET /modelo/info           │    │  ← Informações do modelo
│  │  POST /predict              │    │  ← Faz previsão
│  │  GET /predict/latest        │    │  ← Previsão com dados atuais
│  └─────────────────────────────┘    │
│                │                    │
│                ▼                    │
│  ┌─────────────────────────────┐    │
│  │  Modelo LSTM + Scaler       │    │
│  │  (carregados na memória)    │    │
│  └─────────────────────────────┘    │
└─────────────────────────────────────┘
```

## 1. Setup e Verificação

In [1]:
# =============================================================================
# VERIFICAR ARQUIVOS NECESSÁRIOS
# =============================================================================

import os

arquivos_necessarios = [
    '../models/lstm_vale3.keras',
    '../models/scaler.joblib',
    '../models/metricas.json',
    '../data/processed/config.json'
]

print("VERIFICAÇÃO DE ARQUIVOS")
print("=" * 50)

todos_ok = True
for arquivo in arquivos_necessarios:
    existe = os.path.exists(arquivo)
    status = "✓" if existe else "✗ FALTANDO!"
    print(f"  {status} {arquivo}")
    if not existe:
        todos_ok = False

if todos_ok:
    print("\n✓ Todos os arquivos encontrados!")
else:
    print("\n⚠️ Execute os notebooks anteriores primeiro!")

VERIFICAÇÃO DE ARQUIVOS
  ✓ ../models/lstm_vale3.keras
  ✓ ../models/scaler.joblib
  ✓ ../models/metricas.json
  ✓ ../data/processed/config.json

✓ Todos os arquivos encontrados!


In [2]:
# =============================================================================
# IMPORTS
# =============================================================================

import numpy as np
import json
from datetime import datetime, timedelta

# TensorFlow
import tensorflow as tf

# Scaler
import joblib

# yfinance para buscar dados atuais
import yfinance as yf

# pandas para manipulação
import pandas as pd

print("✓ Imports realizados com sucesso!")

✓ Imports realizados com sucesso!


## 2. Carregar Modelo e Configurações

Vamos testar o carregamento antes de criar a API.

In [3]:
# =============================================================================
# CARREGAR MODELO E SCALER
# =============================================================================

# Carrega o modelo treinado
modelo = tf.keras.models.load_model('../models/lstm_vale3.keras')
print("✓ Modelo carregado!")

# Carrega o scaler
scaler = joblib.load('../models/scaler.joblib')
print("✓ Scaler carregado!")

# Carrega configurações
with open('../data/processed/config.json', 'r') as f:
    config = json.load(f)
print("✓ Configurações carregadas!")

# Carrega métricas
with open('../models/metricas.json', 'r') as f:
    metricas = json.load(f)
print("✓ Métricas carregadas!")

print(f"\nConfiguração do modelo:")
print(f"  Janela temporal: {config['janela_temporal']} dias")
print(f"  Input shape: {config['input_shape']}")

✓ Modelo carregado!
✓ Scaler carregado!
✓ Configurações carregadas!
✓ Métricas carregadas!

Configuração do modelo:
  Janela temporal: 60 dias
  Input shape: [60, 1]


In [4]:
# =============================================================================
# FUNÇÃO DE PREVISÃO
# =============================================================================

def fazer_previsao(precos_historicos: list[float]) -> dict:
    """
    Faz previsão do próximo preço baseado nos últimos N dias.
    
    Parâmetros:
    -----------
    precos_historicos : list[float]
        Lista com os últimos 60 preços de fechamento (em R$)
    
    Retorna:
    --------
    dict com:
        - preco_previsto: previsão em R$
        - confianca: indicador de confiança baseado nas métricas
    """
    janela = config['janela_temporal']
    
    # Validação
    if len(precos_historicos) != janela:
        raise ValueError(f"Esperado {janela} preços, recebido {len(precos_historicos)}")
    
    # Converte para numpy array
    precos = np.array(precos_historicos).reshape(-1, 1)
    
    # Normaliza usando o mesmo scaler do treino
    precos_norm = scaler.transform(precos)
    
    # Reshape para formato LSTM: (1, 60, 1)
    entrada = precos_norm.reshape(1, janela, 1)
    
    # Faz previsão
    previsao_norm = modelo.predict(entrada, verbose=0)
    
    # Desnormaliza para R$
    previsao_reais = scaler.inverse_transform(previsao_norm)
    
    preco_previsto = float(previsao_reais[0, 0])
    
    # Calcula variação em relação ao último preço
    ultimo_preco = precos_historicos[-1]
    variacao = ((preco_previsto - ultimo_preco) / ultimo_preco) * 100
    
    return {
        'preco_previsto': round(preco_previsto, 2),
        'ultimo_preco': round(ultimo_preco, 2),
        'variacao_percentual': round(variacao, 2),
        'direcao': 'alta' if variacao > 0 else 'baixa' if variacao < 0 else 'estável',
        'mae_modelo': metricas['teste']['mae_reais'],
        'mape_modelo': metricas['teste']['mape']
    }

print("✓ Função fazer_previsao definida!")

✓ Função fazer_previsao definida!


In [5]:
# =============================================================================
# FUNÇÃO PARA BUSCAR DADOS ATUAIS
# =============================================================================

def buscar_dados_recentes(ticker: str = 'VALE3.SA', dias: int = 60) -> dict:
    """
    Busca os dados mais recentes do Yahoo Finance.
    
    Parâmetros:
    -----------
    ticker : str
        Código da ação (default: VALE3.SA)
    dias : int
        Quantidade de dias a buscar (default: 60)
    
    Retorna:
    --------
    dict com:
        - precos: lista dos preços de fechamento
        - datas: lista das datas correspondentes
        - ultima_data: data mais recente
    """
    # Busca mais dias do que precisa (para garantir dias úteis suficientes)
    data_fim = datetime.now()
    data_inicio = data_fim - timedelta(days=dias * 2)
    
    # Download dos dados
    df = yf.download(
        ticker,
        start=data_inicio.strftime('%Y-%m-%d'),
        end=data_fim.strftime('%Y-%m-%d'),
        progress=False
    )
    
    # Trata MultiIndex se necessário
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    
    # Pega os últimos N dias
    df = df.tail(dias)
    
    if len(df) < dias:
        raise ValueError(f"Dados insuficientes. Esperado {dias}, obtido {len(df)}")
    
    precos = df['Close'].values.tolist()
    datas = [d.strftime('%Y-%m-%d') for d in df.index]
    
    return {
        'precos': [float(p) for p in precos],
        'datas': datas,
        'ultima_data': datas[-1],
        'ticker': ticker
    }

print("✓ Função buscar_dados_recentes definida!")

✓ Função buscar_dados_recentes definida!


In [6]:
# =============================================================================
# TESTE DAS FUNÇÕES
# =============================================================================

print("TESTE DE PREVISÃO COM DADOS REAIS")
print("=" * 50)

# Busca dados recentes
print("\n1. Buscando dados recentes da VALE3...")
dados = buscar_dados_recentes('VALE3.SA', 60)
print(f"   ✓ {len(dados['precos'])} dias obtidos")
print(f"   Última data: {dados['ultima_data']}")
print(f"   Último preço: R$ {dados['precos'][-1]:.2f}")

# Faz previsão
print("\n2. Fazendo previsão...")
resultado = fazer_previsao(dados['precos'])

print(f"\n" + "=" * 50)
print("RESULTADO DA PREVISÃO")
print("=" * 50)
print(f"\n   Último preço real:  R$ {resultado['ultimo_preco']:.2f}")
print(f"   Previsão próximo:   R$ {resultado['preco_previsto']:.2f}")
print(f"   Variação esperada:  {resultado['variacao_percentual']:+.2f}%")
print(f"   Direção:            {resultado['direcao'].upper()}")
print(f"\n   Erro médio do modelo (MAE): R$ {resultado['mae_modelo']:.2f}")
print(f"   Erro percentual (MAPE):     {resultado['mape_modelo']:.2f}%")

TESTE DE PREVISÃO COM DADOS REAIS

1. Buscando dados recentes da VALE3...


/Users/mariaaraujo/Documents/tech-challenge-4/venv/lib/python3.11/site-packages/yfinance/scrapers/history.py:201: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()


   ✓ 60 dias obtidos
   Última data: 2026-01-23
   Último preço: R$ 85.02

2. Fazendo previsão...

RESULTADO DA PREVISÃO

   Último preço real:  R$ 85.02
   Previsão próximo:   R$ 80.66
   Variação esperada:  -5.12%
   Direção:            BAIXA

   Erro médio do modelo (MAE): R$ 1.58
   Erro percentual (MAPE):     8.24%


## 3. Criar a API FastAPI

Agora vamos criar o arquivo da API que será usado no deploy.

In [7]:
# =============================================================================
# CRIAR ESTRUTURA DE PASTAS PARA API
# =============================================================================

os.makedirs('../src/api', exist_ok=True)

print("✓ Pasta src/api criada!")

✓ Pasta src/api criada!


In [8]:
%%writefile ../src/api/main.py
"""
API FastAPI para previsão de preços da VALE3 usando LSTM.

Endpoints:
- GET  /                  → Informações gerais da API
- GET  /health            → Health check
- GET  /modelo/info       → Informações do modelo
- POST /predict           → Previsão com dados fornecidos
- GET  /predict/latest    → Previsão com dados mais recentes
"""

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import List, Optional
import numpy as np
import tensorflow as tf
import joblib
import json
import os
from datetime import datetime, timedelta
import yfinance as yf
import pandas as pd

# =============================================================================
# CONFIGURAÇÃO DE CAMINHOS
# =============================================================================

# Detecta se está rodando local ou no Lambda
if os.path.exists('/var/task'):
    # AWS Lambda
    BASE_PATH = '/var/task'
else:
    # Local - ajusta caminho relativo
    BASE_PATH = os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

MODEL_PATH = os.path.join(BASE_PATH, 'models', 'lstm_vale3.keras')
SCALER_PATH = os.path.join(BASE_PATH, 'models', 'scaler.joblib')
CONFIG_PATH = os.path.join(BASE_PATH, 'data', 'processed', 'config.json')
METRICS_PATH = os.path.join(BASE_PATH, 'models', 'metricas.json')

# =============================================================================
# INICIALIZAÇÃO DA API
# =============================================================================

app = FastAPI(
    title="API LSTM VALE3",
    description="API para previsão de preços da VALE3 usando modelo LSTM",
    version="1.0.0",
    docs_url="/docs",
    redoc_url="/redoc"
)

# =============================================================================
# CARREGAMENTO DO MODELO (uma única vez na inicialização)
# =============================================================================

print(f"Carregando modelo de: {MODEL_PATH}")

try:
    modelo = tf.keras.models.load_model(MODEL_PATH)
    scaler = joblib.load(SCALER_PATH)
    
    with open(CONFIG_PATH, 'r') as f:
        config = json.load(f)
    
    with open(METRICS_PATH, 'r') as f:
        metricas = json.load(f)
    
    MODELO_CARREGADO = True
    print("✓ Modelo carregado com sucesso!")
except Exception as e:
    print(f"✗ Erro ao carregar modelo: {e}")
    MODELO_CARREGADO = False
    modelo = None
    scaler = None
    config = {'janela_temporal': 60}
    metricas = {}

# =============================================================================
# SCHEMAS (Pydantic)
# =============================================================================

class PrecosInput(BaseModel):
    """Schema para entrada de preços históricos."""
    precos: List[float] = Field(
        ...,
        description="Lista com os últimos 60 preços de fechamento em R$",
        min_items=60,
        max_items=60
    )
    
    class Config:
        schema_extra = {
            "example": {
                "precos": [55.0 + i * 0.1 for i in range(60)]
            }
        }

class PrevisaoOutput(BaseModel):
    """Schema para saída da previsão."""
    preco_previsto: float = Field(..., description="Preço previsto para o próximo dia em R$")
    ultimo_preco: float = Field(..., description="Último preço utilizado na previsão")
    variacao_percentual: float = Field(..., description="Variação percentual esperada")
    direcao: str = Field(..., description="Direção esperada: alta, baixa ou estável")
    mae_modelo: float = Field(..., description="Erro médio absoluto do modelo em R$")
    mape_modelo: float = Field(..., description="Erro percentual médio do modelo")
    data_previsao: str = Field(..., description="Data/hora da previsão")
    ticker: str = Field(default="VALE3.SA", description="Ticker da ação")

class HealthOutput(BaseModel):
    """Schema para health check."""
    status: str
    modelo_carregado: bool
    timestamp: str

class ModeloInfoOutput(BaseModel):
    """Schema para informações do modelo."""
    nome: str
    versao: str
    janela_temporal: int
    metricas_teste: dict
    hiperparametros: dict

# =============================================================================
# FUNÇÕES AUXILIARES
# =============================================================================

def fazer_previsao(precos_historicos: List[float]) -> dict:
    """Faz previsão do próximo preço."""
    if not MODELO_CARREGADO:
        raise HTTPException(status_code=503, detail="Modelo não carregado")
    
    janela = config['janela_temporal']
    
    if len(precos_historicos) != janela:
        raise HTTPException(
            status_code=400, 
            detail=f"Esperado {janela} preços, recebido {len(precos_historicos)}"
        )
    
    # Prepara dados
    precos = np.array(precos_historicos).reshape(-1, 1)
    precos_norm = scaler.transform(precos)
    entrada = precos_norm.reshape(1, janela, 1)
    
    # Previsão
    previsao_norm = modelo.predict(entrada, verbose=0)
    previsao_reais = scaler.inverse_transform(previsao_norm)
    
    preco_previsto = float(previsao_reais[0, 0])
    ultimo_preco = precos_historicos[-1]
    variacao = ((preco_previsto - ultimo_preco) / ultimo_preco) * 100
    
    return {
        'preco_previsto': round(preco_previsto, 2),
        'ultimo_preco': round(ultimo_preco, 2),
        'variacao_percentual': round(variacao, 2),
        'direcao': 'alta' if variacao > 0 else 'baixa' if variacao < 0 else 'estável',
        'mae_modelo': round(metricas.get('teste', {}).get('mae_reais', 0), 2),
        'mape_modelo': round(metricas.get('teste', {}).get('mape', 0), 2),
        'data_previsao': datetime.now().isoformat(),
        'ticker': 'VALE3.SA'
    }

def buscar_dados_recentes(ticker: str = 'VALE3.SA', dias: int = 60) -> dict:
    """Busca dados recentes do Yahoo Finance."""
    data_fim = datetime.now()
    data_inicio = data_fim - timedelta(days=dias * 2)
    
    try:
        df = yf.download(
            ticker,
            start=data_inicio.strftime('%Y-%m-%d'),
            end=data_fim.strftime('%Y-%m-%d'),
            progress=False
        )
        
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        
        df = df.tail(dias)
        
        if len(df) < dias:
            raise HTTPException(
                status_code=500,
                detail=f"Dados insuficientes. Esperado {dias}, obtido {len(df)}"
            )
        
        return {
            'precos': [float(p) for p in df['Close'].values.tolist()],
            'ultima_data': df.index[-1].strftime('%Y-%m-%d'),
            'ticker': ticker
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Erro ao buscar dados: {str(e)}")

# =============================================================================
# ENDPOINTS
# =============================================================================

@app.get("/", tags=["Geral"])
async def root():
    """Endpoint raiz com informações gerais da API."""
    return {
        "nome": "API LSTM VALE3",
        "descricao": "API para previsão de preços da VALE3 usando modelo LSTM",
        "versao": "1.0.0",
        "endpoints": {
            "health": "/health",
            "modelo_info": "/modelo/info",
            "previsao_manual": "POST /predict",
            "previsao_automatica": "GET /predict/latest",
            "documentacao": "/docs"
        }
    }

@app.get("/health", response_model=HealthOutput, tags=["Sistema"])
async def health_check():
    """Verifica se a API está funcionando corretamente."""
    return {
        "status": "healthy" if MODELO_CARREGADO else "unhealthy",
        "modelo_carregado": MODELO_CARREGADO,
        "timestamp": datetime.now().isoformat()
    }

@app.get("/modelo/info", response_model=ModeloInfoOutput, tags=["Modelo"])
async def modelo_info():
    """Retorna informações sobre o modelo treinado."""
    if not MODELO_CARREGADO:
        raise HTTPException(status_code=503, detail="Modelo não carregado")
    
    return {
        "nome": "LSTM VALE3",
        "versao": "1.0.0",
        "janela_temporal": config['janela_temporal'],
        "metricas_teste": {
            "mae_reais": metricas.get('teste', {}).get('mae_reais', 0),
            "rmse_reais": metricas.get('teste', {}).get('rmse_reais', 0),
            "mape": metricas.get('teste', {}).get('mape', 0)
        },
        "hiperparametros": metricas.get('hiperparametros', {})
    }

@app.post("/predict", response_model=PrevisaoOutput, tags=["Previsão"])
async def predict(dados: PrecosInput):
    """
    Faz previsão do próximo preço baseado nos preços fornecidos.
    
    Envie uma lista com exatamente 60 preços de fechamento (em R$).
    """
    resultado = fazer_previsao(dados.precos)
    return resultado

@app.get("/predict/latest", response_model=PrevisaoOutput, tags=["Previsão"])
async def predict_latest(ticker: str = "VALE3.SA"):
    """
    Faz previsão usando os dados mais recentes do Yahoo Finance.
    
    Este endpoint busca automaticamente os últimos 60 dias de dados.
    """
    # Busca dados
    dados = buscar_dados_recentes(ticker, config['janela_temporal'])
    
    # Faz previsão
    resultado = fazer_previsao(dados['precos'])
    resultado['ticker'] = ticker
    
    return resultado

# =============================================================================
# HANDLER PARA AWS LAMBDA (Mangum)
# =============================================================================

# Importação condicional do Mangum para AWS Lambda
try:
    from mangum import Mangum
    handler = Mangum(app)
except ImportError:
    # Mangum não instalado (desenvolvimento local)
    handler = None

Writing ../src/api/main.py


In [9]:
print("✓ Arquivo main.py criado em src/api/")

✓ Arquivo main.py criado em src/api/


In [10]:
%%writefile ../src/api/__init__.py
"""API FastAPI para previsão de preços VALE3."""

Writing ../src/api/__init__.py


## 4. Testar a API Localmente

### Como rodar a API

Abra um terminal na pasta do projeto e execute:

```bash
cd src/api
uvicorn main:app --reload --host 0.0.0.0 --port 8000
```

A API estará disponível em:
- **Swagger UI**: http://localhost:8000/docs
- **ReDoc**: http://localhost:8000/redoc

---

### Testando com Python (requests)

In [ ]:
# =============================================================================
# CÓDIGO PARA TESTAR A API (execute após subir o servidor)
# =============================================================================

# Descomente e execute após iniciar a API com uvicorn

'''
import requests

BASE_URL = "http://localhost:8000"

# Teste 1: Health Check
print("1. Health Check:")
response = requests.get(f"{BASE_URL}/health")
print(f"   Status: {response.json()}")

# Teste 2: Info do Modelo
print("\n2. Info do Modelo:")
response = requests.get(f"{BASE_URL}/modelo/info")
print(f"   {response.json()}")

# Teste 3: Previsão com dados atuais
print("\n3. Previsão com dados mais recentes:")
response = requests.get(f"{BASE_URL}/predict/latest")
resultado = response.json()
print(f"   Preço previsto: R$ {resultado['preco_previsto']}")
print(f"   Variação: {resultado['variacao_percentual']}%")
print(f"   Direção: {resultado['direcao']}")
'''

print("Código de teste preparado!")
print("Descomente e execute após iniciar a API com:")
print("  uvicorn main:app --reload --host 0.0.0.0 --port 8000")

## 5. Preparar para Deploy na AWS

### Estrutura para AWS Lambda com Container

Vamos criar os arquivos necessários para deploy.

In [ ]:
%%writefile ../Dockerfile
# =============================================================================
# Dockerfile para AWS Lambda com TensorFlow
# =============================================================================

# Imagem base otimizada para Lambda com Python
FROM public.ecr.aws/lambda/python:3.11

# Variáveis de ambiente
ENV PYTHONDONTWRITEBYTECODE=1
ENV PYTHONUNBUFFERED=1

# Copia requirements primeiro (para cache de camadas)
COPY requirements.txt ${LAMBDA_TASK_ROOT}/

# Instala dependências
RUN pip install --no-cache-dir -r requirements.txt

# Copia o código da aplicação
COPY src/ ${LAMBDA_TASK_ROOT}/src/
COPY models/ ${LAMBDA_TASK_ROOT}/models/
COPY data/processed/config.json ${LAMBDA_TASK_ROOT}/data/processed/

# Define o handler
CMD ["src.api.main.handler"]

In [ ]:
%%writefile ../requirements-deploy.txt
# =============================================================================
# Requirements para Deploy (versões fixas para estabilidade)
# =============================================================================

# API
fastapi==0.109.0
uvicorn==0.27.0
mangum==0.17.0
pydantic==2.5.3

# ML
tensorflow==2.15.0
numpy==1.26.3
scikit-learn==1.4.0
joblib==1.3.2

# Dados
pandas==2.1.4
yfinance==0.2.36

In [ ]:
%%writefile ../scripts/deploy_aws.sh
#!/bin/bash
# =============================================================================
# Script de Deploy para AWS Lambda
# =============================================================================

# Configurações - ALTERE CONFORME NECESSÁRIO
AWS_REGION="us-east-1"
AWS_ACCOUNT_ID="SEU_ACCOUNT_ID"
ECR_REPO_NAME="lstm-vale3-api"
LAMBDA_FUNCTION_NAME="lstm-vale3-predict"

# Monta nome completo do repositório ECR
ECR_URI="${AWS_ACCOUNT_ID}.dkr.ecr.${AWS_REGION}.amazonaws.com/${ECR_REPO_NAME}"

echo "========================================"
echo "Deploy API LSTM VALE3 para AWS Lambda"
echo "========================================"

# 1. Login no ECR
echo "\n1. Fazendo login no ECR..."
aws ecr get-login-password --region $AWS_REGION | docker login --username AWS --password-stdin $ECR_URI

# 2. Criar repositório ECR (se não existir)
echo "\n2. Criando repositório ECR (se necessário)..."
aws ecr create-repository --repository-name $ECR_REPO_NAME --region $AWS_REGION 2>/dev/null || echo "Repositório já existe"

# 3. Build da imagem Docker
echo "\n3. Construindo imagem Docker..."
docker build -t $ECR_REPO_NAME .

# 4. Tag da imagem
echo "\n4. Tagueando imagem..."
docker tag $ECR_REPO_NAME:latest $ECR_URI:latest

# 5. Push para ECR
echo "\n5. Enviando imagem para ECR..."
docker push $ECR_URI:latest

# 6. Atualizar Lambda (se já existir) ou criar
echo "\n6. Atualizando função Lambda..."
aws lambda update-function-code \
    --function-name $LAMBDA_FUNCTION_NAME \
    --image-uri $ECR_URI:latest \
    --region $AWS_REGION 2>/dev/null || \
echo "Função não existe. Crie manualmente no console AWS ou use 'aws lambda create-function'"

echo "\n========================================"
echo "Deploy concluído!"
echo "========================================"

In [ ]:
# Torna o script executável
os.makedirs('../scripts', exist_ok=True)
os.chmod('../scripts/deploy_aws.sh', 0o755)
print("✓ Script de deploy criado!")

## 6. Instruções de Deploy

### Opção 1: Rodar Localmente (Desenvolvimento)

```bash
# Na pasta raiz do projeto
cd src/api
uvicorn main:app --reload --port 8000
```

Acesse: http://localhost:8000/docs

---

### Opção 2: Deploy AWS Lambda (Produção)

#### Pré-requisitos:
1. AWS CLI configurado (`aws configure`)
2. Docker instalado
3. Conta AWS com permissões para ECR e Lambda

#### Passos:

```bash
# 1. Edite o script com suas configurações
nano scripts/deploy_aws.sh

# 2. Execute o deploy
./scripts/deploy_aws.sh

# 3. Configure API Gateway no console AWS
#    - Crie uma nova API HTTP
#    - Conecte à função Lambda
#    - Deploy para obter a URL pública
```

---

### Opção 3: Deploy Alternativo (Mais Simples)

Se preferir algo mais simples que AWS Lambda:

- **Render.com**: Deploy gratuito com Docker
- **Railway.app**: Deploy simples com GitHub
- **Fly.io**: Containers globais
- **EC2**: VM tradicional na AWS

## 7. Resumo dos Arquivos Criados

In [ ]:
# =============================================================================
# RESUMO FINAL
# =============================================================================

print("=" * 60)
print("RESUMO - API FASTAPI")
print("=" * 60)

print("\n📁 ARQUIVOS CRIADOS:")
print("""  
   tech-challenge-4/
   ├── src/
   │   └── api/
   │       ├── __init__.py
   │       └── main.py          ← API FastAPI
   ├── scripts/
   │   └── deploy_aws.sh        ← Script de deploy
   ├── Dockerfile               ← Para AWS Lambda
   └── requirements-deploy.txt  ← Dependências fixas
""")

print("\n🔗 ENDPOINTS DA API:")
print("   GET  /                → Informações gerais")
print("   GET  /health          → Health check")
print("   GET  /modelo/info     → Métricas e hiperparâmetros")
print("   POST /predict         → Previsão com dados manuais")
print("   GET  /predict/latest  → Previsão com dados atuais")

print("\n🚀 PARA RODAR LOCALMENTE:")
print("   cd src/api")
print("   uvicorn main:app --reload --port 8000")
print("   Acesse: http://localhost:8000/docs")

print("\n" + "=" * 60)
print("PRÓXIMOS PASSOS")
print("=" * 60)
print("\n1. Testar API localmente")
print("2. Fazer deploy (AWS Lambda ou alternativa)")
print("3. Documentar a URL final")
print("4. Gravar vídeo de demonstração")
print("5. Preparar entrega do Tech Challenge")